# Sprint 2 — Cleaning & Prep

Reload the raw CICIDS 2017 flows fresh from `data/raw/`. A new notebook means a new kernel, so it cannot see `df` from `01_eda.ipynb`; we rebuild it from the immutable raw data each run.

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import re
from glob import glob

dataset = glob('../data/raw/MachineLearningCVE/*.csv')
df = pd.concat((pd.read_csv(i) for i in dataset), ignore_index=True)
df.shape

(2830743, 79)

## 1. Clean column names

Strip whitespace, lowercase, and collapse runs of non-alphanumeric characters (spaces, `/`, `.`) into a single `_`. Note: ` Fwd Header Length` appears **twice** in the raw file — they become `fwd_header_length` and `fwd_header_length_1` (a duplicate column dropped in step 4).

In [2]:
def clean_name(name):
    # strip, lowercase, collapse any run of non-alphanumerics into a single _
    return re.sub(r'[^0-9a-zA-Z]+', '_', name.strip()).strip('_').lower()

df.columns = [clean_name(c) for c in df.columns]
assert df.columns.is_unique, df.columns[df.columns.duplicated()].tolist()
df.columns.tolist()

['destination_port',
 'flow_duration',
 'total_fwd_packets',
 'total_backward_packets',
 'total_length_of_fwd_packets',
 'total_length_of_bwd_packets',
 'fwd_packet_length_max',
 'fwd_packet_length_min',
 'fwd_packet_length_mean',
 'fwd_packet_length_std',
 'bwd_packet_length_max',
 'bwd_packet_length_min',
 'bwd_packet_length_mean',
 'bwd_packet_length_std',
 'flow_bytes_s',
 'flow_packets_s',
 'flow_iat_mean',
 'flow_iat_std',
 'flow_iat_max',
 'flow_iat_min',
 'fwd_iat_total',
 'fwd_iat_mean',
 'fwd_iat_std',
 'fwd_iat_max',
 'fwd_iat_min',
 'bwd_iat_total',
 'bwd_iat_mean',
 'bwd_iat_std',
 'bwd_iat_max',
 'bwd_iat_min',
 'fwd_psh_flags',
 'bwd_psh_flags',
 'fwd_urg_flags',
 'bwd_urg_flags',
 'fwd_header_length',
 'bwd_header_length',
 'fwd_packets_s',
 'bwd_packets_s',
 'min_packet_length',
 'max_packet_length',
 'packet_length_mean',
 'packet_length_std',
 'packet_length_variance',
 'fin_flag_count',
 'syn_flag_count',
 'rst_flag_count',
 'psh_flag_count',
 'ack_flag_count',
 'ur

## 2. Binarize the label

Collapse the 15-class label into `benign` (0) vs `malicious` (1). `malicious` is the **positive class** so recall = detection rate and a false negative = a missed intrusion. The original multi-class `label` is kept.

In [3]:
# malicious = 1, benign = 0  (original multi-class 'label' is preserved)
df['label_binary'] = (df['label'] != 'BENIGN').astype(int)
df['label_text'] = df['label_binary'].map({0: 'benign', 1: 'malicious'})

df[['label', 'label_binary', 'label_text']].value_counts()

label                       label_binary  label_text
BENIGN                      0             benign        2273097
DoS Hulk                    1             malicious      231073
PortScan                    1             malicious      158930
DDoS                        1             malicious      128027
DoS GoldenEye               1             malicious       10293
FTP-Patator                 1             malicious        7938
SSH-Patator                 1             malicious        5897
DoS slowloris               1             malicious        5796
DoS Slowhttptest            1             malicious        5499
Bot                         1             malicious        1966
Web Attack � Brute Force    1             malicious        1507
Web Attack � XSS            1             malicious         652
Infiltration                1             malicious          36
Web Attack � Sql Injection  1             malicious          21
Heartbleed                  1             malicious

## 3. Infinities & rate-column NaN

Every inf/NaN here traces to **zero-duration flows**:
- `0 bytes / 0 duration = 0/0 = NaN` → true rate is `0` (nothing moved). Constant fill, no leakage, done now.
- `>0 / 0 duration = +inf` → **marked as NaN now**, then filled with the **training-set** max *after the split* (leak-safe).

Expect `flow_bytes_s` (1509) and `flow_packets_s` (2867) to still hold NaN after this cell — intentional; they are the marked infinities awaiting the post-split fill.

In [4]:
# (a) zero-activity flows (0/0) -> true rate 0. Only flow_bytes_s has these.
df['flow_bytes_s'] = df['flow_bytes_s'].fillna(0)

# (b) divide-by-zero-duration (+inf) -> mark NaN now; fill with TRAIN max after the split (leak-safe)
df[['flow_bytes_s', 'flow_packets_s']] = df[['flow_bytes_s', 'flow_packets_s']].replace([np.inf, -np.inf], np.nan)

# check: no infinities left anywhere; NaN should remain ONLY in the two rate cols (the marked infs)
num = df.select_dtypes('number')
print('infinities remaining:', int(np.isinf(num).to_numpy().sum()))
print('NaN by column:')
print(df.isna().sum()[lambda s: s > 0])

infinities remaining: 0
NaN by column:
flow_bytes_s      1509
flow_packets_s    2867
dtype: int64


## 4. Drop redundant & constant columns

Six groups of byte-identical columns. **8 are constant** (all-zero — the Bulk features + `bwd_psh_flags`/`bwd_urg_flags`): zero variance, no signal → drop all. **5 are exact duplicates** of another column under a second name → keep one, drop the copy. Drops 13 columns (79 → 66 of the original).

In [5]:
drop_cols = [
    # constant (zero variance -> no signal)
    'bwd_psh_flags', 'bwd_urg_flags',
    'fwd_avg_bytes_bulk', 'fwd_avg_packets_bulk', 'fwd_avg_bulk_rate',
    'bwd_avg_bytes_bulk', 'bwd_avg_packets_bulk', 'bwd_avg_bulk_rate',
    # exact duplicate of another column (keep the first name)
    'subflow_fwd_packets',   # == total_fwd_packets
    'subflow_bwd_packets',   # == total_backward_packets
    'syn_flag_count',        # == fwd_psh_flags
    'cwe_flag_count',        # == fwd_urg_flags
    'fwd_header_length_1',   # == fwd_header_length
]
df = df.drop(columns=drop_cols)
df.shape

(2830743, 68)

## 5. Repair impossible negatives

`init_win_bytes_*` = `-1` is a legitimate *"no TCP window"* sentinel → **kept** (~2.44M rows). The 115 negative-`flow_duration` rows are all benign → **dropped**. Every *other* impossible negative — including 194 malicious rows (4 of 11 Heartbleed, 1 of 36 Infiltration) — is **clipped to 0** (leak-free), so no rare attacks are lost.

In [6]:
# init_win_bytes_* (-1) is a legitimate "no window" sentinel -> leave untouched.

# (a) drop the 115 negative-duration rows (all benign -> safe)
df = df[df['flow_duration'] >= 0].copy()

# (b) repair every other impossible negative by clipping to 0 (keeps all rows, incl. 194 malicious)
clip_cols = ['flow_iat_min', 'flow_iat_mean', 'flow_iat_max', 'fwd_iat_min',
             'fwd_header_length', 'bwd_header_length', 'min_seg_size_forward']
df[clip_cols] = df[clip_cols].clip(lower=0)

# verify: no impossible negatives remain (sentinels excluded); rare attacks retained
chk = df.select_dtypes('number').drop(columns=['init_win_bytes_forward', 'init_win_bytes_backward'])
print('rows:', len(df), '| impossible negatives:', int((chk < 0).to_numpy().sum()))
print('Heartbleed kept:', int((df['label'] == 'Heartbleed').sum()))
df.shape

rows: 2830628 | impossible negatives: 0
Heartbleed kept: 11


(2830628, 68)

## 6. Train/test split

The **leakage boundary**: after this line, every *fitted* step (the rate-column max fill, scaling, resampling) is computed on **train only**, then applied to test.

- `X` = the 65 features (all three label columns removed — leaving any in would hand the model the answer)
- `y` = `label_binary`
- **stratify on the multi-class `label`** so all 15 classes (incl. rare attacks) appear in both splits
- `test_size=0.2`, `random_state=42` for reproducibility

In [7]:
from sklearn.model_selection import train_test_split

# X = features only (drop all label columns); y = the binary target
label_cols = ['label', 'label_binary', 'label_text']
X = df.drop(columns=label_cols)
y = df['label_binary']

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    stratify=df['label'],   # multi-class -> rare attacks land in BOTH train and test
    random_state=42,
)

print('train:', X_train.shape, '| test:', X_test.shape)
print('malicious frac  ->  train: %.4f   test: %.4f' % (y_train.mean(), y_test.mean()))
# NOTE: flow_bytes_s / flow_packets_s still carry their marked-inf NaN here;
# they get filled with the TRAIN max in the next (post-split) step.

train: (2264502, 65) | test: (566126, 65)
malicious frac  ->  train: 0.1970   test: 0.1970


## 7. Rate-column max-fill (the deferred fitted step)

The `+inf → NaN` marks from step 3 (`flow_bytes_s`, `flow_packets_s`) are genuine high-rate flows — bytes/packets over a duration that rounds to 0. Fill them with the column **max**.

A max is a **fitted** statistic, so it belongs on the train-only side of the split (`foundation.md Section 7 #5`, `Section 7 #10`): learn it from `X_train`, then apply the *same* value to `X_test`. A global or test-derived max would leak the test set's most extreme flow into the fill.

In [8]:
# Fill the marked-inf rate NaNs with the column MAX, fitted on TRAIN only (foundation.md Section 7 #5, Section 7 #10).
rate_cols = ['flow_bytes_s', 'flow_packets_s']

# how many NaNs we're about to fill (should match step 3: flow_bytes_s 1509, flow_packets_s 2867)
filled = X_train[rate_cols].isna().sum() + X_test[rate_cols].isna().sum()

train_max = X_train[rate_cols].max()               # .max() skips NaN -> max over the finite train rates
X_train[rate_cols] = X_train[rate_cols].fillna(train_max)
X_test[rate_cols]  = X_test[rate_cols].fillna(train_max)   # test gets the TRAIN max, never its own

# self-check (ml-practices.md Section 4): both splits must be NaN-free in these cols now
print('filled (train + test):')
print(filled.to_string())
print('\nfill value = train max:')
print(train_max.to_string())
print('\nNaN remaining ->  train:', int(X_train[rate_cols].isna().to_numpy().sum()),
      ' test:', int(X_test[rate_cols].isna().to_numpy().sum()))

filled (train + test):
flow_bytes_s      1509
flow_packets_s    2867

fill value = train max:
flow_bytes_s      2.071000e+09
flow_packets_s    4.000000e+06

NaN remaining ->  train: 0  test: 0


## 8. Persist the split

Save `X_train / X_test / y_train / y_test` to `data/processed/*.parquet` so the modeling notebooks (`03+`) load the **identical** split instead of re-cleaning 1.7 GB from raw — this is what keeps the DT-vs-SVM comparison fair (`foundation.md Section 7 #9`). `data/processed/` is git-ignored and regenerable; the cell reloads each file to confirm the round-trip.

In [ ]:
# 8. Persist the split -> data/processed/ (parquet) so Sprint 3 (DT) and Sprint 5 (SVM)
#    load the *identical* rows without re-parsing raw each session (foundation.md Section 7 #9, Section 10).
from pathlib import Path

out = Path('../data/processed')
out.mkdir(parents=True, exist_ok=True)
X_train.to_parquet(out / 'X_train.parquet')
X_test.to_parquet(out / 'X_test.parquet')
y_train.to_frame().to_parquet(out / 'y_train.parquet')
y_test.to_frame().to_parquet(out / 'y_test.parquet')

# self-check (ml-practices.md Section 4): reload with pd.read_parquet -- the same call 03 will use --
# and confirm shapes + class balance survive the round-trip
for f in sorted(out.glob('*.parquet')):
    print(f'{f.name:16} {pd.read_parquet(f).shape}')
print('reloaded malicious frac -> train: %.4f  test: %.4f' % (
    pd.read_parquet(out / 'y_train.parquet')['label_binary'].mean(),
    pd.read_parquet(out / 'y_test.parquet')['label_binary'].mean()))